# 🎓 AI-based Personalized Learning Habit Analysis System

**Mục tiêu:** Phân tích thói quen học tập của sinh viên và dự đoán kết quả học tập.

**Dataset:** Student Performance Analytics Dataset (10,000 mẫu)

**Pipeline:**
1. Load & Khám phá dữ liệu (EDA)
2. Tiền xử lý dữ liệu
3. Huấn luyện 3 mô hình: Linear Regression, Random Forest, Gradient Boosting
4. Ensemble: Stacking Regressor
5. Đánh giá & So sánh
6. Gợi ý thời gian học tối ưu

## 📦 Bước 1: Import thư viện

In [ ]:
# ── Thư viện cơ bản ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Trực quan hoá ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Tiền xử lý ───────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── Mô hình ──────────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import StackingRegressor

# ── Đánh giá ─────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Thiết lập style đồ thị ───────────────────────────────────────────────────
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Import thư viện thành công!')

## 📂 Bước 2: Load dữ liệu

In [ ]:
# Đường dẫn tới file dữ liệu
DATA_PATH = r'D:\ML_project\Student Performance Analytics Dataset.csv'

# Đọc dữ liệu vào DataFrame
df = pd.read_csv(DATA_PATH)

print(f'📊 Kích thước dataset: {df.shape[0]} mẫu × {df.shape[1]} cột')
print(f'\n📋 Các cột:')
print(df.dtypes)
print(f'\n🔍 5 dòng đầu:')
df.head()

## 🔍 Bước 3: Khám phá dữ liệu (EDA)

In [ ]:
# ── Thống kê mô tả ────────────────────────────────────────────────────────────
print('📈 Thống kê mô tả:')
df.describe().round(2)

In [ ]:
# ── Kiểm tra giá trị thiếu ────────────────────────────────────────────────────
missing = df.isnull().sum()
print('❓ Giá trị thiếu:')
print(missing[missing > 0] if missing.any() else '✅ Không có giá trị thiếu!')

In [ ]:
# ── Phân phối biến mục tiêu (overall_score) ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram của overall_score
axes[0].hist(df['overall_score'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Phân phối Overall Score')
axes[0].set_xlabel('Overall Score')
axes[0].set_ylabel('Số lượng')

# Phân phối theo Grade
grade_labels = {0: 'F', 1: 'D', 2: 'C', 3: 'B', 4: 'A'}
grade_counts = df['grade'].map(grade_labels).value_counts().reindex(['A','B','C','D','F'])
axes[1].bar(grade_counts.index, grade_counts.values, color=['#2ecc71','#3498db','#f39c12','#e67e22','#e74c3c'])
axes[1].set_title('Số lượng sinh viên theo Xếp loại')
axes[1].set_xlabel('Xếp loại')
axes[1].set_ylabel('Số lượng')

plt.tight_layout()
plt.show()

In [ ]:
# ── Mối tương quan giữa các feature và overall_score ─────────────────────────
numeric_cols = df.select_dtypes(include=np.number).columns.drop(['student_id', 'grade'])
corr_with_target = df[numeric_cols].corr()['overall_score'].drop('overall_score').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr_with_target.values]
corr_with_target.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Tương quan giữa các đặc trưng và Overall Score')
ax.set_ylabel('Hệ số tương quan (Pearson)')
ax.set_xlabel('')
ax.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\n🔗 Tương quan với overall_score:')
print(corr_with_target.round(3))

In [ ]:
# ── Scatter: Study Hours vs Overall Score theo giới tính ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Study hours vs overall score
for gender, label, color in [(0, 'Nam', '#3498db'), (1, 'Nữ', '#e91e8c')]:
    subset = df[df['gender'] == gender]
    axes[0].scatter(subset['study_hours_per_day'], subset['overall_score'],
                    alpha=0.3, s=15, label=label, color=color)
axes[0].set_xlabel('Giờ học mỗi ngày')
axes[0].set_ylabel('Điểm tổng (Overall Score)')
axes[0].set_title('Giờ học vs Điểm tổng')
axes[0].legend()

# Sleep hours vs overall score
axes[1].scatter(df['sleep_hours'], df['overall_score'],
                alpha=0.3, s=15, color='#9b59b6')
axes[1].set_xlabel('Giờ ngủ mỗi ngày')
axes[1].set_ylabel('Điểm tổng (Overall Score)')
axes[1].set_title('Giờ ngủ vs Điểm tổng')

plt.tight_layout()
plt.show()

## ⚙️ Bước 4: Tiền xử lý dữ liệu

In [ ]:
# ── Chọn features và target ───────────────────────────────────────────────────
# Loại bỏ: student_id (ID không có ý nghĩa), grade (được tính từ overall_score)
# Target: overall_score (điểm tổng kết - biến cần dự đoán)

FEATURE_COLS = [
    'gender',                # Giới tính
    'study_hours_per_day',   # Số giờ học mỗi ngày ← thói quen học tập
    'attendance_percentage', # Tỉ lệ tham dự lớp
    'assignment_score',      # Điểm bài tập
    'midterm_score',         # Điểm giữa kỳ
    'participation_score',   # Điểm tham gia
    'internet_access',       # Có Internet không
    'extra_classes',         # Có học thêm không
    'parent_education',      # Trình độ học vấn cha mẹ
    'sleep_hours',           # Số giờ ngủ
]
TARGET_COL = 'overall_score'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print(f'✅ Features: {len(FEATURE_COLS)} cột')
print(f'✅ Target  : {TARGET_COL}')
print(f'✅ Kích thước X: {X.shape}, y: {y.shape}')

In [ ]:
# ── Chia tập Train / Test (80 / 20) ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% để test
    random_state=42      # Đặt seed để tái tạo kết quả
)

print(f'🔀 Tập huấn luyện : {X_train.shape[0]} mẫu')
print(f'🔀 Tập kiểm tra   : {X_test.shape[0]} mẫu')

In [ ]:
# ── Chuẩn hoá dữ liệu (StandardScaler) ───────────────────────────────────────
# StandardScaler: đưa dữ liệu về mean=0, std=1
# Quan trọng cho Linear Regression nhưng cũng tốt cho các mô hình khác

scaler = StandardScaler()

# Fit scaler trên tập train, sau đó transform cả train và test
# ⚠️ Không được fit trên test → tránh data leakage!
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Chuẩn hoá dữ liệu hoàn tất!')
print(f'   Mean của X_train sau scale: {X_train_scaled.mean(axis=0).round(6)}')

## 🤖 Bước 5: Huấn luyện mô hình

In [ ]:
# ── Hàm tiện ích: Đánh giá mô hình ──────────────────────────────────────────
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """
    Huấn luyện mô hình và tính các chỉ số đánh giá.
    Trả về dict chứa tên và các metric.
    """
    # Huấn luyện
    model.fit(X_tr, y_tr)

    # Dự đoán
    y_pred = model.predict(X_te)

    # Tính metric
    mae  = mean_absolute_error(y_te, y_pred)
    mse  = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_te, y_pred)

    # Cross-validation (5-fold) trên train
    cv_r2 = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()

    result = {
        'Model': name,
        'MAE' : round(mae, 4),
        'MSE' : round(mse, 4),
        'RMSE': round(rmse, 4),
        'R²'  : round(r2, 4),
        'CV R²': round(cv_r2, 4),
        '_model': model,
        '_y_pred': y_pred,
    }

    print(f'\n🔹 {name}')
    print(f'   MAE  = {mae:.4f}  (sai số tuyệt đối trung bình)')
    print(f'   MSE  = {mse:.4f}  (sai số bình phương trung bình)')
    print(f'   RMSE = {rmse:.4f}')
    print(f'   R²   = {r2:.4f}  (hệ số xác định, cao hơn = tốt hơn)')
    print(f'   CV R²= {cv_r2:.4f} (cross-val 5-fold trên train)')

    return result

results = []   # Danh sách lưu kết quả tất cả mô hình
print('✅ Hàm đánh giá sẵn sàng!')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MÔ HÌNH 1: LINEAR REGRESSION
# Ý tưởng: Tìm đường thẳng (hyperplane) tốt nhất fit với dữ liệu.
# Ưu điểm: Đơn giản, dễ hiểu, huấn luyện nhanh.
# Nhược điểm: Không nắm bắt được quan hệ phi tuyến.
# ═══════════════════════════════════════════════════════════════════

lr_model = LinearRegression(
    fit_intercept=True   # Cho phép có hệ số tự do (bias)
)

res_lr = evaluate_model(
    'Linear Regression',
    lr_model,
    X_train_scaled, y_train,
    X_test_scaled,  y_test
)
results.append(res_lr)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MÔ HÌNH 2: RANDOM FOREST REGRESSOR
# Ý tưởng: Tập hợp nhiều cây quyết định (Decision Trees),
#          mỗi cây được train trên tập con ngẫu nhiên của data.
# Ưu điểm: Chống overfitting tốt, xử lý được quan hệ phi tuyến.
# Nhược điểm: Chậm hơn, khó giải thích hơn LR.
# ═══════════════════════════════════════════════════════════════════

rf_model = RandomForestRegressor(
    n_estimators=200,    # Số cây trong rừng
    max_depth=12,        # Độ sâu tối đa mỗi cây (kiểm soát overfitting)
    min_samples_split=5, # Số mẫu tối thiểu để chia nhánh
    random_state=42,
    n_jobs=-1            # Dùng tất cả CPU để tăng tốc
)

res_rf = evaluate_model(
    'Random Forest',
    rf_model,
    X_train_scaled, y_train,
    X_test_scaled,  y_test
)
results.append(res_rf)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MÔ HÌNH 3: GRADIENT BOOSTING REGRESSOR
# Ý tưởng: Xây dựng mô hình theo thứ tự (tuần tự),
#          mỗi cây mới cố gắng sửa lỗi của cây trước.
# Ưu điểm: Thường đạt accuracy cao nhất trong 3 mô hình.
# Nhược điểm: Cần tuning hyperparameter cẩn thận.
# ═══════════════════════════════════════════════════════════════════

gb_model = GradientBoostingRegressor(
    n_estimators=200,    # Số vòng boosting
    learning_rate=0.08,  # Tốc độ học (nhỏ hơn → ổn định hơn, cần nhiều vòng hơn)
    max_depth=5,         # Độ sâu mỗi cây (thường dùng 3-6)
    subsample=0.8,       # Dùng 80% dữ liệu mỗi vòng (giảm overfitting)
    random_state=42
)

res_gb = evaluate_model(
    'Gradient Boosting',
    gb_model,
    X_train_scaled, y_train,
    X_test_scaled,  y_test
)
results.append(res_gb)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MÔ HÌNH 4 (ENSEMBLE): STACKING REGRESSOR
#
# Ý tưởng Stacking:
#   Tầng 1 (base learners): LR, RF, GB dự đoán độc lập.
#   Tầng 2 (meta-learner) : Ridge Regression học cách
#                           kết hợp tối ưu các dự đoán trên.
#
#   Input → [LR, RF, GB] → [dự đoán 1, dự đoán 2, dự đoán 3]
#                                         ↓
#                                   Ridge (meta) → Kết quả cuối
#
# Ưu điểm: Tận dụng ưu điểm của cả 3 mô hình.
# ═══════════════════════════════════════════════════════════════════

# Định nghĩa base learners (dùng lại các model đã có)
base_learners = [
    ('lr', LinearRegression()),
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)),
]

# Meta-learner: Ridge Regression (Linear Regression có regularization)
meta_learner = Ridge(alpha=1.0)

stacking_model = StackingRegressor(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,                # Cross-validation để tạo predictions cho meta-learner
    passthrough=False,   # Chỉ dùng predictions của base, không dùng thêm features gốc
    n_jobs=-1
)

res_stack = evaluate_model(
    'Stacking Ensemble',
    stacking_model,
    X_train_scaled, y_train,
    X_test_scaled,  y_test
)
results.append(res_stack)

## 📊 Bước 6: So sánh tổng hợp các mô hình

In [ ]:
# ── Bảng so sánh ─────────────────────────────────────────────────────────────
comparison_df = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith('_')}
    for r in results
]).set_index('Model')

# Highlight mô hình tốt nhất (R² cao nhất)
best_model_name = comparison_df['R²'].idxmax()

print('=' * 60)
print('         SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH')
print('=' * 60)
print(comparison_df.to_string())
print('=' * 60)
print(f'🏆 Mô hình tốt nhất (R² cao nhất): {best_model_name}')
print('   (MAE/MSE thấp = sai số nhỏ; R² cao = giải thích được nhiều variance)')

In [ ]:
# ── Biểu đồ so sánh ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

model_names = comparison_df.index.tolist()
colors      = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']

# ── 1. MAE so sánh ───────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(model_names, comparison_df['MAE'], color=colors, edgecolor='white', linewidth=1.2)
ax1.set_title('MAE (thấp hơn = tốt hơn)', fontweight='bold')
ax1.set_ylabel('Mean Absolute Error')
for bar, val in zip(bars, comparison_df['MAE']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax1.set_xticks(range(len(model_names)))
ax1.set_xticklabels(model_names, rotation=15, ha='right')

# ── 2. RMSE so sánh ──────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
bars2 = ax2.bar(model_names, comparison_df['RMSE'], color=colors, edgecolor='white', linewidth=1.2)
ax2.set_title('RMSE (thấp hơn = tốt hơn)', fontweight='bold')
ax2.set_ylabel('Root Mean Squared Error')
for bar, val in zip(bars2, comparison_df['RMSE']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax2.set_xticks(range(len(model_names)))
ax2.set_xticklabels(model_names, rotation=15, ha='right')

# ── 3. R² so sánh ────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
bars3 = ax3.bar(model_names, comparison_df['R²'], color=colors, edgecolor='white', linewidth=1.2)
ax3.set_title('R² Score (cao hơn = tốt hơn)', fontweight='bold')
ax3.set_ylabel('R² Score')
ax3.set_ylim(0, 1.05)
for bar, val in zip(bars3, comparison_df['R²']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax3.set_xticks(range(len(model_names)))
ax3.set_xticklabels(model_names, rotation=15, ha='right')

# ── 4. Radar chart: tổng hợp ─────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
# Normalize metrics (0-1, cao = tốt)
mae_norm  = 1 - (comparison_df['MAE']  / comparison_df['MAE'].max())
rmse_norm = 1 - (comparison_df['RMSE'] / comparison_df['RMSE'].max())
r2_norm   = comparison_df['R²']
cv_norm   = comparison_df['CV R²']

x_pos = np.arange(4)
width = 0.18
for i, (name, c) in enumerate(zip(model_names, colors)):
    vals = [mae_norm.iloc[i], rmse_norm.iloc[i], r2_norm.iloc[i], cv_norm.iloc[i]]
    ax4.bar(x_pos + i * width, vals, width, label=name, color=c, alpha=0.85)

ax4.set_title('Tổng hợp (cao = tốt, đã chuẩn hoá)', fontweight='bold')
ax4.set_xticks(x_pos + width * 1.5)
ax4.set_xticklabels(['1-MAE', '1-RMSE', 'R²', 'CV R²'])
ax4.legend(fontsize=8, loc='lower right')
ax4.set_ylim(0, 1.1)

fig.suptitle('So sánh Hiệu năng Các Mô hình ML\n(Student Performance Prediction)',
             fontsize=14, fontweight='bold')
plt.savefig(r'D:\ML_project\model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Biểu đồ đã lưu: D:/ML_project/model_comparison.png')

In [ ]:
# ── Biểu đồ Actual vs Predicted (mô hình tốt nhất) ───────────────────────────
best_result = max(results, key=lambda r: r['R²'])
y_pred_best = best_result['_y_pred']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: Actual vs Predicted
axes[0].scatter(y_test, y_pred_best, alpha=0.35, s=18, color='#3498db', label='Dự đoán')
lim = [y_test.min() - 2, y_test.max() + 2]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Lý tưởng (y=x)')
axes[0].set_xlabel('Điểm thực tế')
axes[0].set_ylabel('Điểm dự đoán')
axes[0].set_title(f'Actual vs Predicted\n({best_result["Model"]})')
axes[0].legend()

# Phân phối residuals (sai số)
residuals = y_test.values - y_pred_best
axes[1].hist(residuals, bins=40, color='#e67e22', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Sai số (Actual − Predicted)')
axes[1].set_ylabel('Số lượng')
axes[1].set_title('Phân phối sai số (Residuals)')

plt.tight_layout()
plt.show()

print(f'\nSai số trung bình: {residuals.mean():.4f} (gần 0 = mô hình không bị bias)')
print(f'Std sai số       : {residuals.std():.4f}')

In [ ]:
# ── Feature Importance (Random Forest) ───────────────────────────────────────
# Random Forest cung cấp điểm quan trọng của từng feature

importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature'   : FEATURE_COLS,
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors_fi = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_imp_df)))
ax.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color=colors_fi)
ax.set_xlabel('Mức độ quan trọng')
ax.set_title('Feature Importance (Random Forest)\n(Đặc trưng nào ảnh hưởng nhiều nhất đến kết quả)')

for i, (val, name) in enumerate(zip(feat_imp_df['Importance'], feat_imp_df['Feature'])):
    ax.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\n📌 Top 3 đặc trưng quan trọng nhất:')
print(feat_imp_df.sort_values('Importance', ascending=False).head(3).to_string(index=False))

## 💡 Bước 7: Gợi ý thời gian học tối ưu

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# GỢI Ý THỜI GIAN HỌC TỐI ƯU
# Phân tích: điểm trung bình theo số giờ học → tìm ngưỡng tối ưu
# ═══════════════════════════════════════════════════════════════════

# Chia study_hours thành bins (khoảng 0.5 giờ)
df['study_bin'] = pd.cut(df['study_hours_per_day'], bins=16)
study_analysis  = df.groupby('study_bin', observed=True)['overall_score'].agg(
    mean_score='mean', count='size'
).reset_index()

study_analysis['mid_hours'] = study_analysis['study_bin'].apply(lambda x: x.mid)

# Tìm khoảng giờ học cho điểm cao nhất
best_bin   = study_analysis.loc[study_analysis['mean_score'].idxmax()]
optimal_hr = float(best_bin['mid_hours'])

fig, ax = plt.subplots(figsize=(11, 5))

# Vẽ đường mean score
ax.plot(study_analysis['mid_hours'], study_analysis['mean_score'],
        marker='o', markersize=6, color='#2ecc71', linewidth=2, label='Điểm TB')
ax.fill_between(study_analysis['mid_hours'], study_analysis['mean_score'],
                alpha=0.15, color='#2ecc71')

# Đánh dấu điểm tối ưu
ax.axvline(optimal_hr, color='red', linestyle='--', linewidth=1.8,
           label=f'Tối ưu ≈ {optimal_hr:.1f} giờ/ngày')
ax.annotate(f'Tối ưu\n{optimal_hr:.1f}h\nScore={best_bin["mean_score"]:.1f}',
            xy=(optimal_hr, best_bin['mean_score']),
            xytext=(optimal_hr + 0.5, best_bin['mean_score'] - 6),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

ax.set_xlabel('Số giờ học mỗi ngày')
ax.set_ylabel('Điểm tổng trung bình')
ax.set_title('Phân tích: Số giờ học vs Điểm tổng\n(Tìm thời gian học tối ưu)')
ax.legend()
plt.tight_layout()
plt.show()

# Phân tích thêm: Top/Bottom performers học bao nhiêu giờ
top_students    = df[df['overall_score'] >= df['overall_score'].quantile(0.75)]
bottom_students = df[df['overall_score'] <= df['overall_score'].quantile(0.25)]

print('\n📊 Phân tích giờ học theo nhóm sinh viên:')
print(f'  👑 Top 25% sinh viên      : TB {top_students["study_hours_per_day"].mean():.2f} giờ/ngày')
print(f'  📉 Bottom 25% sinh viên   : TB {bottom_students["study_hours_per_day"].mean():.2f} giờ/ngày')
print(f'  🎯 Điểm tối ưu (model)    : ~{optimal_hr:.1f} giờ/ngày')

## 🎯 Bước 8: Dự đoán cho sinh viên mới

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DỰ ĐOÁN CHO SINH VIÊN CỤ THỂ
# Nhập thông tin sinh viên → dự đoán điểm tổng + tư vấn
# ═══════════════════════════════════════════════════════════════════

def predict_student(student_info: dict, model, scaler, feature_cols):
    """
    Dự đoán điểm tổng và đưa ra tư vấn cho một sinh viên.

    Parameters:
        student_info : dict - thông tin sinh viên
        model        : mô hình đã train
        scaler       : scaler đã fit
        feature_cols : danh sách tên feature

    Returns:
        predicted_score (float)
    """
    # Tạo DataFrame 1 hàng từ thông tin sinh viên
    X_new = pd.DataFrame([student_info])[feature_cols]

    # Chuẩn hoá bằng scaler đã train
    X_new_scaled = scaler.transform(X_new)

    # Dự đoán
    predicted_score = model.predict(X_new_scaled)[0]
    predicted_score = np.clip(predicted_score, 0, 100)  # Clamp vào [0, 100]

    # Xếp loại dự đoán
    if   predicted_score >= 85: grade_pred = 'A (Xuất sắc)'
    elif predicted_score >= 70: grade_pred = 'B (Tốt)'
    elif predicted_score >= 55: grade_pred = 'C (Trung bình)'
    elif predicted_score >= 40: grade_pred = 'D (Yếu)'
    else:                       grade_pred = 'F (Không đạt)'

    # Tư vấn dựa trên thói quen học
    tips = []
    if student_info['study_hours_per_day'] < 3:
        tips.append('⚠️  Bạn học dưới 3 giờ/ngày. Hãy tăng lên ít nhất 4-6 giờ.')
    if student_info['attendance_percentage'] < 75:
        tips.append('⚠️  Tỉ lệ tham dự lớp thấp (<75%). Hãy đi học đầy đủ hơn.')
    if student_info['sleep_hours'] < 6:
        tips.append('⚠️  Bạn ngủ ít hơn 6 giờ. Ngủ đủ giúp tăng khả năng ghi nhớ.')
    if student_info['sleep_hours'] > 9:
        tips.append('⚠️  Bạn ngủ trên 9 giờ. Cân bằng thời gian ngủ và học tập.')
    if not tips:
        tips.append('✅ Thói quen học tập của bạn rất tốt! Hãy duy trì.')

    # In kết quả
    print('=' * 50)
    print('     KẾT QUẢ DỰ ĐOÁN')
    print('=' * 50)
    print(f'  Điểm dự đoán : {predicted_score:.2f} / 100')
    print(f'  Xếp loại     : {grade_pred}')
    print()
    print('  📋 Tư vấn cải thiện:')
    for tip in tips:
        print(f'     {tip}')
    print('=' * 50)

    return predicted_score


# ── Ví dụ sinh viên A (học nhiều, ngủ ít) ────────────────────────────────────
print('\n📌 Sinh viên A (chăm học, mất ngủ):')
student_A = {
    'gender'               : 0,    # Nam
    'study_hours_per_day'  : 6.5,  # 6.5 giờ/ngày
    'attendance_percentage': 90.0, # 90% tham dự
    'assignment_score'     : 78.0, # Điểm bài tập
    'midterm_score'        : 72.0, # Điểm giữa kỳ
    'participation_score'  : 60.0, # Điểm tham gia
    'internet_access'      : 1,    # Có Internet
    'extra_classes'        : 1,    # Có học thêm
    'parent_education'     : 3,    # Học vấn cha mẹ cao
    'sleep_hours'          : 5.0,  # Chỉ ngủ 5 giờ
}
score_A = predict_student(student_A, best_result['_model'], scaler, FEATURE_COLS)

# ── Ví dụ sinh viên B (lười học, nghỉ nhiều) ─────────────────────────────────
print('\n📌 Sinh viên B (ít học, hay nghỉ):')
student_B = {
    'gender'               : 1,    # Nữ
    'study_hours_per_day'  : 2.0,  # Chỉ 2 giờ/ngày
    'attendance_percentage': 60.0, # 60% tham dự
    'assignment_score'     : 45.0,
    'midterm_score'        : 40.0,
    'participation_score'  : 20.0,
    'internet_access'      : 0,
    'extra_classes'        : 0,
    'parent_education'     : 1,
    'sleep_hours'          : 8.0,
}
score_B = predict_student(student_B, best_result['_model'], scaler, FEATURE_COLS)

## 📝 Tóm tắt kết quả

In [ ]:
# ── Tóm tắt toàn bộ dự án ────────────────────────────────────────────────────
print('=' * 65)
print('   TỔNG KẾT DỰ ÁN: AI-based Personalized Learning Habit Analysis')
print('=' * 65)

print(f'''\n📊 DATASET:
   File    : Student Performance Analytics Dataset
   Mẫu     : {df.shape[0]:,}
   Features: {len(FEATURE_COLS)}
   Target  : overall_score (0-100)
''')

print('📐 CÁC MÔ HÌNH:')
for r in results:
    best_mark = ' ← 🏆 TỐT NHẤT' if r['Model'] == best_model_name else ''
    print(f"   {r['Model']:25s}  MAE={r['MAE']:.3f}  R²={r['R²']:.4f}{best_mark}")

print(f'''\n💡 GỢI Ý THỜI GIAN HỌC:
   - Thời gian tối ưu: ~{optimal_hr:.1f} giờ/ngày
   - Top 25% sinh viên trung bình: {top_students["study_hours_per_day"].mean():.2f} giờ/ngày
   - Tham dự lớp tối thiểu: 80%+
   - Giờ ngủ khuyến nghị : 7-8 giờ/đêm
''')

print('=' * 65)